# 02. 특보 이벤트 수 세기 (event counts)

**이 노트북이 하는 일**

특보 종류(폭염·한파·호우·대설)별로 **연속해서 켜진 날을 하나의 구간**으로 묶고, 같은 기간에 다른 종류 특보가 겹치는지에 따라 **단일 이벤트**와 **복합 이벤트**로 나눈 뒤, 다른 사건과의 겹침을 단계별로 제외해 **분석에 쓸 수 있는 "깨끗한 이벤트" 수**를 세 가지 기준으로 나란히 셉니다 (피드백 1).

**세 가지 기준**

| 기준 | 이벤트 정의 |
|---|---|
| **단일 (주 분석)** | 이벤트 기간(시작~종료) 중 다른 종류 특보가 없음. **해제 후 7일 안에 다른 특보가 시작되면 포함**하되, 회복은 그 특보가 시작하기 전날까지만 보고 그때까지 못 돌아오면 중도절단 (`다음특보시작일`에 기록) |
| **복합 (compound)** | 이벤트 기간 중 다른 종류 특보가 겹침. 겹친 특보들을 하나로 합쳐 시작~종료를 다시 잡고(예: 폭염+호우) 조합별로 집계 |
| **민감도** | 단일 이벤트 중 **종료 후 7일 안에도** 다른 종류 특보가 없는 것 (이전 02번의 "다른 특보 겹침 제외" 기준과 같음) |

**공통 규칙**

- 지속일수 = 종료일 − 시작일 + 1. 기간(train 2014~2023 / validation 2024 / test 2025)은 시작일 기준입니다.
- 다른 사건과의 겹침 확인 창은 시작일부터 **종료일 + `ALERT_MASK_AFTER_DAYS`(7)일**까지이고, 아래 순서로 제외합니다: ① 파일 경계(시작·종료 불명; 특보 파일은 2014-01-08부터 2025-12-31까지) ② 코로나(2020-02-01~2022-04-30)·메르스(2015-05-20~07-28) ③ 공휴일·명절 ±2일·근로자의 날·파업 ④ 계단 더미 시행일(요금 인상, 기후동행카드, 신림선, 신분당선) ±30일.
- 14일을 넘는 이벤트는 `long_event`로 표시합니다 (기준선 예측 기간이 지속일수 + 14일이라 따로 봄).

> 특보는 서울 전체 기준이라 9개 역이 같은 이벤트를 공유합니다 (시청 집회 겹침은 참고용 열). **이용량(`total`)은 쓰지 않고**, 2025(test) 이벤트도 세지만 **모델링에는 07번 전까지 쓰지 않습니다.**

**입력 파일**

- `data/processed/model_input_9stations.csv` (특보 열, 가리기 열, 계단 더미 열)

**출력 파일**

- `results/events.csv` (이벤트별 표: 분류, 종류, 시작일, 종료일, 지속일수, 기간, 겹침 표시, 민감도 포함 여부, long_event)
- `results/event_counts.csv` (기준 × 종류 × 단계 × 기간별 이벤트 수)
- `results/event_criteria_summary.csv` (세 기준을 나란히 놓은 요약)
- `results/event_overlap_reasons.csv` (겹침 사유별 이벤트 수)


**설정 불러오기**: Colab이면 저장소를 clone해 경로를 맞추고, `00_settings.ipynb`의 설정을 불러옵니다 (로컬이면 clone은 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')

# 설정(경로, 역, 열 이름, 기간, 차수 등)을 불러옵니다.
%run ./00_settings.ipynb

필요한 패키지를 불러옵니다.

In [2]:
import numpy as np
import pandas as pd

**날짜별 달력 표 만들기**: 모델 입력 표에서 날짜마다 특보, 가리기 열(공휴일·명절·근로자의 날·코로나·메르스·파업), 계단 더미를 가져옵니다. 이 열들은 서울 전체 기준이라 9개 역이 같은 값이어야 하므로 같은지 확인하고, 시청 행을 대표로 씁니다.

In [3]:
mi = pd.read_csv(PROC / 'model_input_9stations.csv', parse_dates=['date'])
same_cols = [c for c in MASK_COLS if c != 'rally_cityhall'] + list(EVENT_TYPES.values()) + ['any_alert']
nun = mi.groupby('date')[same_cols].nunique().max()            # 날짜마다 9개 역의 값이 몇 종류인가
assert (nun <= 1).all(), nun[nun > 1]
cal = mi[mi.station == STATIONS[0]].set_index('date').sort_index()[same_cols + ['rally_cityhall']]
print('날짜', cal.index.min().date(), '~', cal.index.max().date(), f'({len(cal)}일) | 특보 여부를 모르는 날(alert_unknown):', int(cal.alert_unknown.sum()), '일')

날짜 2014-01-01 ~ 2025-12-31 (4383일) | 특보 여부를 모르는 날(alert_unknown): 7 일


**계단 더미 시행일 ±30일 구간**: 더미 열이 처음 1이 되는 날을 시행일로 보고, 그 앞뒤 `EVENT_STEP_PM_DAYS`(30)일을 "겹침 구간"으로 표시합니다 (역별 더미는 그 역의 행에서 찾음).

In [4]:
step_dates = {}
for c in STEP_COLS:
    step_dates[c] = mi.loc[mi[c] == 1, 'date'].min()
for st, c in STATION_STEP.items():
    step_dates[c] = mi.loc[(mi.station == st) & (mi[c] == 1), 'date'].min()
near_step = pd.Series(False, index=cal.index)
for d in step_dates.values():
    near_step |= (cal.index >= d - pd.Timedelta(days=EVENT_STEP_PM_DAYS)) & (cal.index <= d + pd.Timedelta(days=EVENT_STEP_PM_DAYS))
pd.Series({k: str(v.date()) for k, v in step_dates.items()}, name='시행일')

fare_2015         2015-06-27
fare_2023         2023-10-07
fare_2025         2025-06-28
climate_card      2024-01-27
sillim_line       2022-05-28
sinbundang_ext    2022-05-28
Name: 시행일, dtype: object

**`make_runs`**: 한 종류의 특보 열에서 연속해서 켜진 구간을 찾습니다. 특보 여부를 모르는 날(NaN)은 구간을 끊고, 구간의 앞날 또는 뒷날이 모르는 날이거나 표의 끝이면 시작·종료일을 알 수 없는 구간으로 표시합니다.

In [5]:
def make_runs(s):
    """s: 날짜 인덱스의 특보 열(1=켜짐, 0=꺼짐, NaN=모름). 반환: [(시작일, 종료일, 시작불명, 종료불명), ...]"""
    active = (s == 1)
    gid = (active != active.shift(fill_value=False)).cumsum()
    out = []
    for _, g in active[active].groupby(gid[active]):
        a, b = g.index.min(), g.index.max()
        before, after = a - pd.Timedelta(days=1), b + pd.Timedelta(days=1)
        left = (a == s.index.min()) or (before in s.index and pd.isna(s.loc[before]))
        right = (b == s.index.max()) or (after in s.index and pd.isna(s.loc[after]))
        out.append((a, b, bool(left), bool(right)))
    return out

**`period_of`**: 이벤트의 시작일로 기간을 정합니다 (2023-12-31까지 train, 2024년 validation, 2025년 test).

In [6]:
def period_of(d):
    if d <= pd.Timestamp(TRAIN_END):
        return 'train'
    return 'validation' if d.year == VALID else 'test'

**1) 종류별 연속 구간(run) 찾기**: 4종류 특보의 연속 구간을 모두 모읍니다 (같은 종류끼리는 하루라도 꺼지면 다른 구간).

In [7]:
runs = []
for kind, col in EVENT_TYPES.items():
    for (a, b, left, right) in make_runs(cal[col]):
        runs.append({'종류': kind, '시작': a, '종료': b, '시작불명': left, '종료불명': right})
R = pd.DataFrame(runs).sort_values(['시작', '종료', '종류']).reset_index(drop=True)
print('구간', len(R), '개:', R.종류.value_counts().to_dict())

구간 248 개: {'호우': 100, '한파': 56, '폭염': 56, '대설': 36}


**2) 겹치는 구간 묶기 (단일 / 복합 나누기)**: 날짜 범위가 하루라도 겹치는 구간들을 한 묶음으로 합칩니다 (A가 B와, B가 C와 겹치면 모두 한 묶음). **구간이 하나뿐인 묶음 = 단일 이벤트**, **둘 이상이면 복합 이벤트**이며 복합은 합친 구간의 시작~종료를 새 이벤트 기간으로 씁니다. 종류는 조합(예: 폭염+호우)으로 적습니다.

In [8]:
comp, cur_end, cid = [], None, -1
for _, r in R.iterrows():
    if cur_end is None or r.시작 > cur_end:                    # 앞 묶음과 겹치지 않으면 새 묶음
        cid += 1
        cur_end = r.종료
    else:
        cur_end = max(cur_end, r.종료)
    comp.append(cid)
R['묶음'] = comp
KIND_ORDER = list(EVENT_TYPES)
print('묶음', R.묶음.nunique(), '개 | 단일', int((R.groupby('묶음').size() == 1).sum()), '개, 복합', int((R.groupby('묶음').size() > 1).sum()), '개')

묶음 219 개 | 단일 198 개, 복합 21 개


**3) 이벤트 표 만들기**: 묶음마다 이벤트 한 줄을 만들고, ① 해제 후 7일 안에 시작하는 다음 특보(`다음특보시작일`, 회복을 거기까지만 봄), ② 종료 후 7일 안에 **다른 종류** 특보가 있는지(민감도 기준), ③ 확인 창(시작일 ~ 종료일+7일) 안의 겹침 사유를 계산합니다. 제외 순서에서 처음 걸리는 사유를 `제외단계`로 기록합니다.

In [9]:
REASONS = ['파일경계', '코로나·메르스', '공휴일·명절·근로자의날·파업', '계단더미±30일']     # 제외 순서
rows = []
for cid, g in R.groupby('묶음'):
    kinds = [k for k in KIND_ORDER if k in set(g.종류)]
    cls = '단일' if len(g) == 1 else '복합'
    assert cls == '단일' or len(kinds) >= 2, '같은 종류끼리 겹칠 수 없음'
    a, b = g.시작.min(), g.종료.max()
    left, right = bool(g.시작불명.any()), bool(g.종료불명.any())
    # 해제 후 7일 안에 시작하는 다른 묶음의 특보
    nxt = R[(R.묶음 != cid) & (R.시작 > b) & (R.시작 <= b + pd.Timedelta(days=ALERT_MASK_AFTER_DAYS))]
    next_start = nxt.시작.min() if len(nxt) else pd.NaT
    next_kinds = '+'.join(k for k in KIND_ORDER if k in set(nxt[nxt.시작 == next_start].종류)) if len(nxt) else ''
    other_after = bool(len(nxt[~nxt.종류.isin(kinds)]))          # 종료 후 7일 안에 다른 종류 특보가 있는가 (민감도 기준)
    w = slice(a, min(b + pd.Timedelta(days=ALERT_MASK_AFTER_DAYS), cal.index.max()))       # 겹침 확인 창
    win = cal.loc[w]
    f = {'파일경계': left or right,
         '코로나·메르스': bool(win.covid.any() or win.mers.any()),
         '공휴일·명절·근로자의날·파업': bool(win.holiday.any() or win.myeongjeol_pm2.any() or win.labor_day.any() or win.strike.any()),
         '계단더미±30일': bool(near_step.loc[w].any())}
    rows.append({'분류': cls, '종류': '+'.join(kinds), '시작일': a.date(), '종료일': b.date(), '지속일수': (b - a).days + 1, '기간': period_of(a),
                 '구성구간수': len(g), '시작불명': left, '종료불명': right, '코로나': bool(win.covid.any()), '메르스': bool(win.mers.any()), '공휴일': bool(win.holiday.any()),
                 '명절±2': bool(win.myeongjeol_pm2.any()), '근로자의날': bool(win.labor_day.any()), '파업': bool(win.strike.any()), '계단더미±30일': f['계단더미±30일'],
                 '시청집회(참고)': bool(win.rally_cityhall.any()), '기간경계걸침': period_of(b) != period_of(a),
                 '해제후7일내_다른종류특보': other_after, '민감도단일': cls == '단일' and not other_after,
                 '다음특보시작일': next_start.date() if len(nxt) else None, '다음특보종류': next_kinds,
                 'long_event': (b - a).days + 1 > 14,
                 '제외단계': next((i + 1 for i, r_ in enumerate(REASONS) if f[r_]), 0), '깨끗함': not any(f.values()),
                 '구성구간': '; '.join(f'{r_.종류} {r_.시작.date()}~{r_.종료.date()}' for r_ in g.itertuples())})
ev = pd.DataFrame(rows).sort_values(['시작일', '종류']).reset_index(drop=True)
ev.insert(0, '이벤트번호', [f"{c}-{k}-{i:02d}" for c, k, i in zip(ev.분류, ev.종류, ev.groupby(['분류', '종류']).cumcount() + 1)])
ev['제외사유'] = ev.apply(lambda r_: ', '.join(x for x in REASONS if x == '파일경계' and (r_.시작불명 or r_.종료불명)
                                          or x == '코로나·메르스' and (r_.코로나 or r_.메르스)
                                          or x == '공휴일·명절·근로자의날·파업' and (r_.공휴일 or r_['명절±2'] or r_.근로자의날 or r_.파업)
                                          or x == '계단더미±30일' and r_['계단더미±30일']), axis=1)
print('이벤트', len(ev), '개 (단일', int((ev.분류 == '단일').sum()), '/ 복합', int((ev.분류 == '복합').sum()), '/ 민감도 단일', int(ev.민감도단일.sum()), ') | 깨끗한 이벤트', int(ev.깨끗함.sum()), '개')
ev.head(6)

이벤트 219 개 (단일 198 / 복합 21 / 민감도 단일 121 ) | 깨끗한 이벤트 113 개


,이벤트번호,분류,종류,시작일,종료일,지속일수,기간,구성구간수,시작불명,종료불명,...,기간경계걸침,해제후7일내_다른종류특보,민감도단일,다음특보시작일,다음특보종류,long_event,제외단계,깨끗함,구성구간,제외사유
0,단일-한파-01,단일,한파,2014-01-08,2014-01-10,3,train,1,True,False,...,False,False,True,None,,False,1,False,한파 2014-01-08~2014-01-10,파일경계
1,단일-대설-01,단일,대설,2014-01-20,2014-01-20,1,train,1,False,False,...,False,False,True,None,,False,0,True,대설 2014-01-20~2014-01-20,
2,단일-대설-02,단일,대설,2014-02-08,2014-02-08,1,train,1,False,False,...,False,False,True,None,,False,0,True,대설 2014-02-08~2014-02-08,
3,단일-호우-01,단일,호우,2014-06-22,2014-06-23,2,train,1,False,False,...,False,False,True,None,,False,0,True,호우 2014-06-22~2014-06-23,
4,단일-폭염-01,단일,폭염,2014-07-09,2014-07-11,3,train,1,False,False,...,False,True,False,2014-07-18,호우,False,0,True,폭염 2014-07-09~2014-07-11,
5,단일-호우-02,단일,호우,2014-07-18,2014-07-18,1,train,1,False,False,...,False,False,True,2014-07-22,호우,False,0,True,호우 2014-07-18~2014-07-18,


**이벤트 표 점검**: (1) 모든 구간이 정확히 한 이벤트에 들어갔는지, (2) 이벤트에 들어간 구간의 지속일수 합이 특보가 켜진 날 수와 같은지, (3) 이벤트끼리 날짜가 겹치지 않는지 확인합니다.

In [10]:
assert ev.구성구간수.sum() == len(R)
active_days = {k: int((cal[c] == 1).sum()) for k, c in EVENT_TYPES.items()}
run_days = (R.종료 - R.시작).dt.days.add(1).groupby(R.종류).sum().to_dict()
print('특보 켜진 날 수:', active_days)
print('구간 지속일수 합:', run_days)
assert all(active_days[k] == run_days[k] for k in EVENT_TYPES)
e2 = ev.sort_values('시작일')
assert (pd.to_datetime(e2.시작일.iloc[1:].values) > pd.to_datetime(e2.종료일.iloc[:-1].values)).all(), '이벤트끼리 날짜가 겹침'
print('점검 통과')

특보 켜진 날 수: {'폭염': 386, '한파': 172, '호우': 150, '대설': 48}
구간 지속일수 합: {'대설': 48, '폭염': 386, '한파': 172, '호우': 150}
점검 통과


**이벤트 표 저장**: `results/events.csv` (한 줄이 이벤트 하나, `분류`가 단일/복합, `민감도단일`이 True면 민감도 기준에도 포함).

In [11]:
RESULTS.mkdir(exist_ok=True)
ev.to_csv(RESULTS / 'events.csv', index=False, encoding='utf-8-sig')
print('events.csv:', ev.shape)

events.csv: (219, 28)


**기준별 단계별 이벤트 수**: 세 기준(단일 / 민감도 / 복합)마다 종류(또는 조합) × 기간별로, 전체 이벤트에서 시작해 겹침 사유를 한 단계씩 누적해서 제외한 뒤 남는 수를 셉니다. 마지막 행이 "깨끗한 이벤트"입니다. `train+val`은 07번 전에 모델링에 쓸 수 있는 train과 validation의 합입니다.

In [12]:
STEP_LABELS = ['0 전체 이벤트', '1 파일 경계(시작·종료 불명) 제외', '2 + 코로나·메르스 겹침 제외', '3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외', '4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)']
PERIODS = ['train', 'validation', 'test']
CRITERIA = {'단일(주 분석)': ev.분류 == '단일', '민감도(종료 후 7일 안 다른 특보 제외)': ev.민감도단일, '복합': ev.분류 == '복합'}
rows = []
for crit, sel in CRITERIA.items():
    sub = ev[sel]
    for kind in sorted(sub.종류.unique(), key=lambda s: (s.count('+'), [KIND_ORDER.index(x) for x in s.split('+')])):
        g = sub[sub.종류 == kind]
        for k, lab in enumerate(STEP_LABELS):
            keep = g[(g.제외단계 == 0) | (g.제외단계 > k)]          # k단계까지 제외하고 남는 이벤트
            cnt = keep.기간.value_counts().reindex(PERIODS, fill_value=0)
            rows.append({'기준': crit, '종류': kind, '단계': lab, **cnt.to_dict(), 'train+val': int(cnt['train'] + cnt['validation']), '합계': int(cnt.sum())})
counts = pd.DataFrame(rows)
counts.to_csv(RESULTS / 'event_counts.csv', index=False, encoding='utf-8-sig')
counts.shape

(55, 8)

**세 기준 나란히 보기**: 종류별로 단일 / 민감도 / 복합의 "전체"와 "깨끗한" 이벤트 수(train+validation)를 한 표에 놓습니다. 복합 이벤트는 조합(예: 폭염+호우)이 따로 줄을 이룹니다. 깨끗한 이벤트가 `MIN_EVENTS`(10)개 미만이면 ⚠(참고용)를 붙입니다.

In [13]:
first, last = STEP_LABELS[0], STEP_LABELS[-1]
def tv(crit, kind, step):
    r = counts[(counts.기준 == crit) & (counts.종류 == kind) & (counts.단계 == step)]
    return int(r['train+val'].iloc[0]) if len(r) else None
kinds_all = KIND_ORDER + sorted(set(counts[counts.기준 == '복합'].종류), key=lambda s: [KIND_ORDER.index(x) for x in s.split('+')])
rows = []
for kind in kinds_all:
    row = {'종류': kind}
    for crit, short in [('단일(주 분석)', '단일'), ('민감도(종료 후 7일 안 다른 특보 제외)', '민감도'), ('복합', '복합')]:
        a, b = tv(crit, kind, first), tv(crit, kind, last)
        row[(short, '전체')] = a
        row[(short, '깨끗')] = None if b is None else (f'{b} ⚠' if b < MIN_EVENTS else str(b))
    rows.append(row)
side = pd.DataFrame(rows).set_index('종류')
side.columns = pd.MultiIndex.from_tuples(side.columns)
side.to_csv(RESULTS / 'event_criteria_summary.csv', encoding='utf-8-sig')
side

단일         민감도          복합      
         전체    깨끗    전체    깨끗    전체    깨끗
종류                                       
폭염     40.0    27  14.0    10   NaN  None
한파     43.0    19  35.0    17   NaN  None
호우     70.0    39  41.0    17   NaN  None
대설     24.0    10  20.0   8 ⚠   NaN  None
폭염+호우   NaN  None   NaN  None  11.0   3 ⚠
한파+호우   NaN  None   NaN  None   1.0   0 ⚠
한파+대설   NaN  None   NaN  None   6.0   0 ⚠

**기준별 단계 표 (단일 / 민감도 / 복합)**: 단계마다 기간별 이벤트 수를 모두 보여 줍니다 (마지막 행의 10개 미만 칸은 ⚠).

In [14]:
def mark(v, is_last):
    return f'{v} ⚠' if (is_last and v < MIN_EVENTS) else str(v)
show = counts.copy()
is_last = show.단계 == last
for c in PERIODS + ['train+val', '합계']:
    show[c] = [mark(v, l) for v, l in zip(counts[c], is_last)]
show.set_index(['기준', '종류', '단계'])

train  \
기준                      종류    단계                                           
단일(주 분석)                폭염    0 전체 이벤트                                38   
                              1 파일 경계(시작·종료 불명) 제외                    38   
                              2 + 코로나·메르스 겹침 제외                       33   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            27   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    26   
                        한파    0 전체 이벤트                                42   
                              1 파일 경계(시작·종료 불명) 제외                    41   
                              2 + 코로나·메르스 겹침 제외                       32   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            19   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    19   
                        호우    0 전체 이벤트                                67   
                              1 파일 경계(시작·종료 불명) 제외                    67   
                              2 + 코로나·메르스 겹침 제외                       48   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            38   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    36   
                        대설    0 전체 이벤트                                19   
                              1 파일 경계(시작·종료 불명) 제외                    19   
                              2 + 코로나·메르스 겹침 제외                       14   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외             9   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   9 ⚠   
민감도(종료 후 7일 안 다른 특보 제외) 폭염    0 전체 이벤트                                13   
                              1 파일 경계(시작·종료 불명) 제외                    13   
                              2 + 코로나·메르스 겹침 제외                       12   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외             9   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   9 ⚠   
                        한파    0 전체 이벤트                                34   
                              1 파일 경계(시작·종료 불명) 제외                    33   
                              2 + 코로나·메르스 겹침 제외                       26   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            17   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    17   
                        호우    0 전체 이벤트                                39   
                              1 파일 경계(시작·종료 불명) 제외                    39   
                              2 + 코로나·메르스 겹침 제외                       25   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외            16   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)    15   
                        대설    0 전체 이벤트                                15   
                              1 파일 경계(시작·종료 불명) 제외                    15   
                              2 + 코로나·메르스 겹침 제외                       11   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외             7   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   7 ⚠   
복합                      폭염+호우 0 전체 이벤트                                 8   
                              1 파일 경계(시작·종료 불명) 제외                     8   
                              2 + 코로나·메르스 겹침 제외                        6   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외             3   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   2 ⚠   
                        한파+호우 0 전체 이벤트                                 1   
                              1 파일 경계(시작·종료 불명) 제외                     1   
                              2 + 코로나·메르스 겹침 제외                        1   
                              3 + 공휴일·명절±2·근로자의 날·파업 겹침 제외             0   
                              4 + 계단 더미 시행일 ±30일 겹침 제외 (= 깨끗한 이벤트)   0 ⚠   
                        한파+대설 0 전체 이벤트                                 6   
                              1 파일 경계(시

**이전 02번 결과와의 일관성 점검**: 민감도 기준은 이전 02번의 "다른 종류 특보 겹침 제외" 기준과 같아야 합니다. 이전 결과(깨끗한 이벤트: 폭염 9/1/0, 한파 17/0/2, 호우 15/2/3, 대설 7/1/3 = train/validation/test)와 비교합니다.

In [15]:
old = {'폭염': (9, 1, 0), '한파': (17, 0, 2), '호우': (15, 2, 3), '대설': (7, 1, 3)}
sens = counts[(counts.기준.str.startswith('민감도')) & (counts.단계 == last)].set_index('종류')[PERIODS]
chk = pd.DataFrame({k: dict(zip(PERIODS, v)) for k, v in old.items()}).T
print('이전 02번과 같음:', bool((sens.reindex(chk.index).values == chk.values).all()))
pd.concat({'이전 02번': chk, '민감도(새)': sens.reindex(chk.index)}, axis=1)

이전 02번과 같음: True


이전 02번                 민감도(새)                
    train validation test  train validation test
폭염      9          1    0      9          1    0
한파     17          0    2     17          0    2
호우     15          2    3     15          2    3
대설      7          1    3      7          1    3

**종류별 판정**: 단일(주 분석)의 깨끗한 이벤트 수(train+validation)가 `MIN_EVENTS`(10)개 이상이면 "충분", 미만이면 "참고용"입니다. test(2025)는 07번 전까지 모델링에 쓰지 않으므로 판정에서 뺍니다.

In [16]:
fin = counts[(counts.기준 == '단일(주 분석)') & (counts.단계 == last)].set_index('종류')
judge = pd.DataFrame({'전체 단일 이벤트': counts[(counts.기준 == '단일(주 분석)') & (counts.단계 == first)].set_index('종류')['합계'],
                      '깨끗한 이벤트(train+val)': fin['train+val'], 'train': fin['train'], 'validation': fin['validation'], 'test(참고)': fin['test']})
judge['판정'] = np.where(judge['깨끗한 이벤트(train+val)'] >= MIN_EVENTS, '충분', '참고용(10개 미만)')
judge

,전체 단일 이벤트,깨끗한 이벤트(train+val),train,validation,test(참고),판정
종류,,,,,,
폭염,43,27,26,1,1,충분
한파,48,19,19,0,3,충분
호우,78,39,36,3,5,충분
대설,29,10,9,1,4,충분


**겹침 사유별 이벤트 수**: 분류(단일/복합) × 종류별로, 각 사유에 걸린 이벤트 수를 셉니다 (한 이벤트가 여러 사유에 걸리면 각각 셈). `results/event_overlap_reasons.csv`로 저장합니다.

In [17]:
flag_cols = ['시작불명', '종료불명', '코로나', '메르스', '공휴일', '명절±2', '근로자의날', '파업', '계단더미±30일', '시청집회(참고)', '해제후7일내_다른종류특보', 'long_event']
reasons = ev.groupby(['분류', '종류'])[flag_cols].sum().astype(int)
reasons.insert(0, '전체', ev.groupby(['분류', '종류']).size())
reasons.to_csv(RESULTS / 'event_overlap_reasons.csv', encoding='utf-8-sig')
reasons

전체  시작불명  종료불명  코로나  메르스  공휴일  명절±2  근로자의날  파업  계단더미±30일  시청집회(참고)  \
분류 종류                                                                          
단일 대설     29     0     0    5    0    9     5      0   0         6         5   
   폭염     43     0     0    5    0   10     1      0   0         2         0   
   한파     48     1     1    9    0   15     6      0   1         2         4   
   호우     78     0     0   18    1   14     5      0   0         5         0   
복합 폭염+호우  13     0     0    1    1    6     1      0   0         3         0   
   한파+대설   7     0     0    5    0    3     0      0   0         0         1   
   한파+호우   1     0     0    0    0    0     0      0   1         1         0   

          해제후7일내_다른종류특보  long_event  
분류 종류                                
단일 대설                 6           0  
   폭염                29           3  
   한파                 9           0  
   호우                33           0  
복합 폭염+호우              0           3  
   한파+대설              0           0  
   한파+호우              0           0

**복합 이벤트 조합 현황**: 어떤 종류가 함께 켜졌는지(조합), 이벤트 수, 평균 지속일수, 깨끗한 이벤트 수를 봅니다.

In [18]:
comp_ev = ev[ev.분류 == '복합']
pd.DataFrame({'이벤트 수': comp_ev.groupby('종류').size(), '평균 지속일수': comp_ev.groupby('종류').지속일수.mean().round(1), '최대 지속일수': comp_ev.groupby('종류').지속일수.max(),
              '깨끗(train+val)': comp_ev[comp_ev.깨끗함 & (comp_ev.기간 != 'test')].groupby('종류').size(), '깨끗(test)': comp_ev[comp_ev.깨끗함 & (comp_ev.기간 == 'test')].groupby('종류').size()}).fillna(0).astype(int, errors='ignore')

,이벤트 수,평균 지속일수,최대 지속일수,깨끗(train+val),깨끗(test)
종류,,,,,
폭염+호우,13,11,39,3,1
한파+대설,7,5,7,0,1
한파+호우,1,2,2,0,0


**long_event (지속일수 14일 초과) 따로 보기**: 기준선 예측 기간이 지속일수 + 14일이라 긴 이벤트는 따로 봅니다. 분류와 깨끗함 여부를 함께 보여 줍니다.

In [19]:
ev[ev.long_event][['이벤트번호', '분류', '종류', '시작일', '종료일', '지속일수', '기간', '깨끗함', '제외사유']].reset_index(drop=True)

,이벤트번호,분류,종류,시작일,종료일,지속일수,기간,깨끗함,제외사유
0,단일-폭염-08,단일,폭염,2016-07-31,2016-08-25,26,train,False,공휴일·명절·근로자의날·파업
1,단일-폭염-15,단일,폭염,2018-07-12,2018-08-18,38,train,False,공휴일·명절·근로자의날·파업
2,복합-폭염+호우-04,복합,폭염+호우,2021-07-10,2021-08-13,35,train,False,"코로나·메르스, 공휴일·명절·근로자의날·파업"
3,복합-폭염+호우-06,복합,폭염+호우,2023-07-25,2023-08-09,16,train,False,공휴일·명절·근로자의날·파업
4,복합-폭염+호우-10,복합,폭염+호우,2024-07-24,2024-08-31,39,validation,False,공휴일·명절·근로자의날·파업
5,단일-폭염-41,단일,폭염,2025-07-21,2025-08-06,17,test,False,계단더미±30일


**깨끗한 이벤트 목록 (단일, 주 분석, train+validation)**: 04번 노트북에서 분석할 이벤트의 입력입니다.

In [20]:
ev[(ev.분류 == '단일') & ev.깨끗함 & (ev.기간 != 'test')][['이벤트번호', '종류', '시작일', '종료일', '지속일수', '기간', '다음특보시작일', '다음특보종류', '민감도단일', 'long_event']].reset_index(drop=True)

,이벤트번호,종류,시작일,종료일,지속일수,기간,다음특보시작일,다음특보종류,민감도단일,long_event
0,단일-대설-01,대설,2014-01-20,2014-01-20,1,train,None,,True,False
1,단일-대설-02,대설,2014-02-08,2014-02-08,1,train,None,,True,False
2,단일-호우-01,호우,2014-06-22,2014-06-23,2,train,None,,True,False
3,단일-폭염-01,폭염,2014-07-09,2014-07-11,3,train,2014-07-18,호우,False,False
4,단일-호우-02,호우,2014-07-18,2014-07-18,1,train,2014-07-22,호우,True,False
...,...,...,...,...,...,...,...,...,...,...
90,단일-폭염-39,폭염,2024-06-19,2024-06-21,3,validation,None,,True,False
91,단일-호우-68,호우,2024-07-02,2024-07-02,1,validation,2024-07-09,호우,True,False
92,단일-호우-69,호우,2024-07-17,2024-07-18,2,validation,2024-07-24,폭염,False,False
93,단일-호우-70,호우,2024-09-20,2024-09-21,2,validation,None,,True,False
